# Dispute-category classifier: training and evaluation (Phase 6.5)

Trains a Logistic Regression on **bge-m3 embeddings** (the same
embedding layer used by indexing and retrieval) to predict which of
the four dispute categories a user's first message belongs to.

**Run from the repo root or from `notebooks/`.** The first run
downloads/loads bge-m3, which can take a few minutes.

Outputs: accuracy, per-category precision/recall/F1, confusion
matrix, confidence histogram, and the saved model at
`src/dialogue/models/category_classifier.pkl`.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix)
from sklearn.model_selection import (
    StratifiedKFold, cross_val_score, train_test_split)

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.config import settings
from src.dialogue.category_classifier import CategoryClassifier
from src.dialogue.slot_schema import DisputeCategory

DATA_PATH = ROOT / "data" / "training_data" / "dispute_categories.csv"
RANDOM_STATE = 42

## 1. Load and check the dataset

In [ ]:
df = pd.read_csv(DATA_PATH)
valid = {c.value for c in DisputeCategory}
assert set(df["category"]) <= valid, "unknown category label"
assert not df["message"].duplicated().any(), "duplicate messages"
print(df["category"].value_counts())
df.sample(5, random_state=RANDOM_STATE)

## 2. Embed with bge-m3

Uses `CategoryClassifier.embed`, which wraps the project's
`EmbeddingManager` (normalised embeddings, same model as retrieval).

In [ ]:
clf = CategoryClassifier()
X = clf.embed(df["message"].tolist())
y = df["category"].to_numpy()
print("Embedding matrix:", X.shape)

## 3. Stratified 80/20 train/test split

With only ~80 examples the test set is small (about 16 messages), so
single-split metrics are noisy. Section 4 adds cross-validation as a
more stable estimate.

In [ ]:
idx = np.arange(len(df))
train_idx, test_idx = train_test_split(
    idx, test_size=0.2, stratify=y, random_state=RANDOM_STATE)
X_train, X_test = X[train_idx], X[test_idx]
y_train, y_test = y[train_idx], y[test_idx]
print("train:", len(train_idx), "test:", len(test_idx))

## 4. Choose regularisation strength C

Chosen by cross-validation on the **training split only**, so the
test set stays untouched.

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True,
                      random_state=RANDOM_STATE)
for C in [0.1, 1, 10, 100]:
    scores = cross_val_score(
        LogisticRegression(C=C, max_iter=1000), X_train, y_train, cv=skf)
    print(f"C={C:>6}: {scores.mean():.3f} +/- {scores.std():.3f}")

## 5. Train

Set `BEST_C` from the table above. Higher C gives more confident
probabilities, which matters because of the confidence threshold.

In [ ]:
BEST_C = 10.0  # update from the cross-validation table above
clf.model.set_params(C=BEST_C)
clf.fit(X_train, y_train)
print("classes:", [c.value for c in clf.classes])

## 6. Test-set metrics

In [ ]:
proba = clf.predict_proba(X_test)
pred = np.array([clf.classes[k].value for k in proba.argmax(axis=1)])
conf = proba.max(axis=1)

print(f"Test accuracy: {accuracy_score(y_test, pred):.3f}\n")
print(classification_report(y_test, pred, digits=3, zero_division=0))

### Confusion matrix

In [ ]:
labels = [c.value for c in clf.classes]
cm = confusion_matrix(y_test, pred, labels=labels)

fig, ax = plt.subplots(figsize=(6, 5))
ax.imshow(cm, cmap="Blues")
ax.set_xticks(range(len(labels)))
ax.set_xticklabels(labels, rotation=45, ha="right")
ax.set_yticks(range(len(labels)))
ax.set_yticklabels(labels)
for i in range(len(labels)):
    for j in range(len(labels)):
        ax.text(j, i, cm[i, j], ha="center", va="center")
ax.set_xlabel("Predicted")
ax.set_ylabel("True")
ax.set_title("Confusion matrix (test set)")
plt.tight_layout()
plt.show()

### Confidence distribution

In [ ]:
correct = pred == y_test
bins = np.linspace(0, 1, 11)
plt.figure(figsize=(6, 4))
plt.hist([conf[correct], conf[~correct]], bins=bins, stacked=True,
         label=["correct", "wrong"], color=["tab:blue", "tab:red"])
plt.axvline(settings.CLASSIFIER_CONFIDENCE_THRESHOLD, color="black",
            linestyle="--", label="threshold")
plt.xlabel("Predicted confidence")
plt.ylabel("Test messages")
plt.title("Confidence distribution (test set)")
plt.legend()
plt.tight_layout()
plt.show()

### Coverage vs accuracy at different thresholds

Coverage = share of messages that get a prediction instead of
`None`. Use this to justify `CLASSIFIER_CONFIDENCE_THRESHOLD`.

In [ ]:
for t in [0.3, 0.4, 0.5, 0.6, 0.7, 0.8]:
    keep = conf >= t
    acc = (pred[keep] == y_test[keep]).mean() if keep.any() else float("nan")
    print(f"threshold {t:.1f}: coverage {keep.mean():.2f}, "
          f"accuracy on covered {acc:.2f}")

### Cross-validated accuracy on all data

More stable than the single 16-message test split.

In [ ]:
cv = cross_val_score(
    LogisticRegression(C=BEST_C, max_iter=1000), X, y, cv=skf)
print(f"5-fold CV accuracy: {cv.mean():.3f} +/- {cv.std():.3f}")

## 7. Save the model

In [ ]:
saved_path = clf.save()
print("Saved to:", saved_path)

## 8. Reload and run inference

Held-out examples first, then messages that are **not** labour
disputes. Note: softmax probabilities are not calibrated for
out-of-scope text, so some off-topic messages may still pass the
threshold. Record what you observe.

In [ ]:
loaded = CategoryClassifier.load(saved_path, embed_fn=clf.embed)
df_test = df.iloc[test_idx].reset_index(drop=True)

print("--- held-out examples ---")
for _, row in df_test.head(8).iterrows():
    p = loaded.predict(row["message"])
    shown = f"{p.category.value} ({p.confidence:.2f})" if p else "None"
    print(f"true={row['category']:<24} pred={shown}")
    print(f"   {row['message']}")

print("\n--- out-of-scope / vague messages ---")
for text in ["Salam, necəsiniz?", "Bakıda bu gün hava necədir?",
             "Mənim işdə problemim var."]:
    p = loaded.predict(text)
    shown = f"{p.category.value} ({p.confidence:.2f})" if p else "None"
    print(f"{text!r} -> {shown}")

## Results for the presentation (fill in after running)

| Metric | Value |
|---|---|
| Training examples | 64 (of 80) |
| Test accuracy (16 msgs) | |
| 5-fold CV accuracy | |
| Per-category F1 | |
| Most confused pair | |
| Chosen C / threshold | |